# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pickle
from collections import defaultdict
from tqdm.notebook import trange, tqdm
from icecream import ic

import pandas as pd
import numpy as np
import scipy
import sklearn
import prince

import seaborn as sns
from matplotlib import pyplot as plt

from plotly import express as px
from plotly import graph_objects as go
import plotly.io as pio

In [3]:
from rsfp.metrics import *
from rsfp.constants import *
from rsfp.data import *
from rsfp.matching import *
from rsfp.dimensionality_reduction import *
from rsfp.utils import *

In [4]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)
pd.options.plotting.backend = "plotly"

# Data

## 2023

In [5]:
# load 2023 datasets
df_voters, df_candidates, df_questions = build_all(verbose=True)

Preprocessed voter DataFrame found.
Cleaned voter DataFrame found.
Cleaned candidate DataFrame found.


In [ ]:
# load uncleaned 2023 voter dataset
df_voters_all = build_voters(clean=False)
df_voters_all

In [6]:
# load recommendation data for L2 distance metric
df_voters = df_voters.load_candidate_voting_recommendations('df_voters_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])
df_candidates = df_candidates.load_candidate_recommendation_counts('df_candidates_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])

In [ ]:
# load list recommendations for L2 distance metric
df_voters = df_voters.load_list_voting_recommendations('df_voters_list_recommendations_10_methods', selected_distance_methods=['L2_sv'])

In [7]:
# load recommendation data for evaluated distance metrics
df_voters = df_voters.load_candidate_voting_recommendations('df_voters_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)
df_candidates = df_candidates.load_candidate_recommendation_counts('df_candidates_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# load list recommendations for evaluated distance metrics
df_voters = df_voters.load_list_voting_recommendations('df_voters_list_recommendations_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [14]:
# add dimensionality reduction columns
df_candidates, obj = add_pca_cols(df_candidates, n_dims=2, return_obj=True)
df_voters = add_pca_cols(df_voters, n_dims=2, obj=obj)

df_candidates, obj = add_ca_cols(df_candidates, n_dims=2, return_obj=True)
df_voters = add_ca_cols(df_voters, n_dims=2, obj=obj)

df_candidates, obj = add_tsvd_cols(df_candidates, n_dims=2, return_obj=True)
df_voters = add_tsvd_cols(df_voters, n_dims=2, obj=obj)

In [ ]:
# plot PCA distribution of candidates
fig = px.scatter(df_candidates[~df_candidates['_party'].isin(['Parteilos', 'Andere'])], x='PCA_0', y='-PCA_1', color='_party', color_discrete_map=PARTY2COLOR)
fig.update_layout(
    xaxis_title='Left - Right',
    yaxis_title='Conservative - Liberal',
    legend=dict(
        title='Party',
        x=1,
        y=1,
        xanchor='left',
        yanchor='top',
        itemsizing='constant',
    )
).update_traces(marker_size=3)


## 2019

In [62]:
# load 2019 datasets
df_voters19, df_candidates19, df_questions19 = build_all19(verbose=True)

No cleaned voter19 DataFrame found. Cleaning starts now.
No cleaned candidate19 DataFrame found. Cleaning starts now.


In [ ]:
# load uncleaned 2019 voter dataset
df_voters19_all = build_voters19(clean=False)

In [6]:
# load recommendation data for L2 distance metric
df_voters19 = df_voters19.load_candidate_voting_recommendations('df_voters19_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])
df_candidates19 = df_candidates19.load_candidate_recommendation_counts('df_candidates19_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])

In [ ]:
# load list recommendations for L2 distance metric
df_voters19 = df_voters19.load_list_voting_recommendations('df_voters19_list_recommendations_10_methods', selected_distance_methods=['L2_sv'])

In [276]:
# load recommendation data for evaluated distance metrics
df_voters19 = df_voters19.load_candidate_voting_recommendations('df_voters19_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)
df_candidates19 = df_candidates19.load_candidate_recommendation_counts('df_candidates19_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# load list recommendations for evaluated distance metrics
df_voters19 = df_voters19.load_list_voting_recommendations('df_voters19_list_recommendations_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# add dimensionality reduction columns
df_candidates19, obj = add_pca_cols(df_candidates19, n_dims=2, return_obj=True)
df_voters19 = add_pca_cols(df_voters19, n_dims=2, obj=obj)

df_candidates19, obj = add_ca_cols(df_candidates19, n_dims=2, return_obj=True)
df_voters19 = add_ca_cols(df_voters19, n_dims=2, obj=obj)

df_candidates19, obj = add_tsvd_cols(df_candidates19, n_dims=2, return_obj=True)
df_voters19 = add_tsvd_cols(df_voters19, n_dims=2, obj=obj)

# Comparison

## Evaluation Metrics

In [13]:
# candidate recommendation count metrics

df_candidates.get_recommendation_metrics()

,method,weighted_mean_diff_spc,gini_spc,as_rec_spc_corr,weighted_mean_diff_top,gini_top,as_rec_top_corr,weighted_mean_diff_spc_zh,gini_spc_zh,as_rec_spc_corr_zh,weighted_mean_diff_top_zh,gini_top_zh,as_rec_top_corr_zh
0,L2_sv,-2.613749,0.474809,-0.470265,-3.442942,0.611276,-0.396868,-2.702216,0.470579,-0.480721,-3.803119,0.639160,-0.395285
1,L1,-1.116006,0.373104,-0.279684,-1.717797,0.512178,-0.268260,-1.253336,0.371525,-0.305263,-2.138377,0.550790,-0.284248
2,AC,0.849142,0.317267,0.255847,0.174669,0.437558,0.033966,0.698358,0.305325,0.207252,-0.315592,0.468519,-0.051090
3,angular,0.703141,0.349119,0.190098,0.088449,0.492975,0.014560,0.437874,0.350769,0.111414,-0.517822,0.536856,-0.071053
4,mahalanobis_unweighted,0.259902,0.523231,0.043938,-0.898829,0.545198,-0.134412,0.456401,0.539485,0.069439,-1.251524,0.566835,-0.160995
5,DM_L1_BONUS,2.454397,0.387275,0.583668,2.447371,0.513661,0.358592,2.326486,0.377544,0.540821,2.137063,0.547763,0.254686
6,DM_HYBRID,1.082152,0.348810,0.292091,1.106863,0.497005,0.172499,0.945874,0.346836,0.244148,0.865435,0.543958,0.105802


In [100]:
# list voting recommendation metrics (accuracy) & disagreements with candidate voting recommendations metrics

df_voters.get_voter_metrics(df_candidates, progress_bar=True)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 43.45it/s]


,method,party_match_accuracy,party_rank_mean,party_rank_frac_mean,party_in_top_10_perc,party_in_top_50_perc,disagreement_top_frac_mean,disagreement_strong_top_frac_mean,disagreement_spc_frac_mean,disagreement_strong_spc_frac_mean
0,L2_sv,0.410401,3.629912,0.103437,0.709014,0.958180,0.199929,0.043322,0.238436,0.077896
1,L1,0.417938,3.551119,0.101331,0.717150,0.959116,0.195892,0.074538,0.232054,0.109658
2,AC,0.357397,3.879289,0.111092,0.681199,0.959785,0.237809,0.118678,0.265102,0.149454
3,angular,0.401825,3.799457,0.108637,0.692742,0.957716,0.191387,0.038658,0.229130,0.069529
4,mahalanobis_unweighted,0.290256,4.910633,0.141837,0.573915,0.944993,0.225961,0.177145,0.264884,0.216444
5,DM_L1_BONUS,0.378836,3.815774,0.109054,0.688761,0.960698,0.211151,0.079062,0.242127,0.112864
6,DM_HYBRID,0.402124,3.722670,0.106268,0.701775,0.958510,0.190295,0.065953,0.227106,0.100545


In [101]:
df_party_popularities = df_voters.get_party_list_popularities()

In [93]:
df_party_popularities

,SP,Grüne,PdA,GLP,EVP,Andere,Parteilos,Mitte,MCG,Lega,FDP,EDU,SVP
pref_party,0.205547,0.106356,0.005536,0.136073,0.014285,0.011399,NaN,0.082025,0.000127,0.000536,0.100711,0.004240,0.072947
top_L2_sv,0.174305,0.137634,0.007085,0.128681,0.044966,0.130276,0.070505,0.136082,0.001017,0.001828,0.073413,0.026951,0.067257
top_L1,0.194779,0.156523,0.009542,0.122271,0.037482,0.115623,0.068255,0.121034,0.001507,0.001731,0.072040,0.023675,0.075538
top_AC,0.212400,0.169635,0.009684,0.098208,0.026406,0.146913,0.085119,0.090147,0.001908,0.001673,0.057902,0.021567,0.078438
top_angular,0.232287,0.208602,0.015676,0.082423,0.027736,0.104189,0.066520,0.075270,0.001435,0.001577,0.077487,0.020431,0.086368
top_mahalanobis_unweighted,0.289370,0.337645,0.004244,0.087908,0.020803,0.056686,0.048930,0.069710,0.000309,0.002023,0.032972,0.007383,0.042016
top_DM_L1_BONUS,0.243359,0.210895,0.014338,0.070698,0.021721,0.122339,0.071748,0.067249,0.001982,0.001416,0.070381,0.018932,0.084940
top_DM_HYBRID,0.223128,0.201542,0.013704,0.088923,0.030068,0.107484,0.065600,0.089316,0.001704,0.001554,0.075058,0.019986,0.081933
top_zh_L2_sv,0.188493,0.100109,0.014080,0.111203,0.004169,0.166276,0.189371,0.092840,NaN,NaN,0.064144,0.028075,0.041240
top_zh_L1,0.221853,0.090846,0.018679,0.112491,0.005409,0.131990,0.197127,0.089157,NaN,NaN,0.065594,0.022037,0.044817


In [96]:
df_party_popularity_diffs = df_voters.get_party_popularity_diffs(df_party_popularities)
df_party_popularity_diffs.loc[:, BIG_PARTIES].style.background_gradient('inferno')

,SP,Grüne,GLP,Mitte,FDP,SVP
L2_sv,-0.053403,-0.067438,0.040265,0.053790,0.002202,-0.012928
L1,-0.032929,-0.048549,0.033856,0.038741,0.000143,-0.004648
AC,-0.015308,-0.035438,0.009792,0.007854,-0.014825,-0.000297
angular,0.014523,0.023014,-0.011143,-0.014462,0.006276,0.009381
mahalanobis_unweighted,0.071606,0.152057,-0.005657,-0.020021,-0.039755,-0.038169
DM_L1_BONUS,0.025595,0.025307,-0.022867,-0.022483,-0.002345,0.007952
DM_HYBRID,0.000784,0.012424,-0.004135,0.006607,0.003847,0.004945


In [121]:
df_party_strength_diffs = []
for c in df_party_visibilities.columns:
    df_party_visibilities_removed = df_party_visibilities.drop(columns=c)
    df_party_strength_diffs.append(df_party_visibilities[c] - df_party_visibilities_removed.median(axis=1))

df_party_strength_diffs = pd.DataFrame(df_party_strength_diffs, index=df_party_visibilities.columns)
df_party_strength_diffs

_party,Andere,EDU,EVP,FDP,GLP,Grüne,Lega,MCG,Mitte,Parteilos,PdA,SP,SVP
Party Strength (L2_sv),0.005438,0.001867,0.024544,0.000069,0.026246,-0.054866,-0.000052,-0.000470,0.060597,0.002023,-0.003464,-0.047643,-0.017773
Party Strength (L1),-0.001807,0.001184,0.014364,-0.001653,0.017515,-0.035596,0.000021,-0.000239,0.038648,0.000724,-0.002021,-0.028252,-0.009340
Party Strength (AC),0.008698,0.000611,-0.001589,-0.002342,-0.000935,-0.012346,-0.000050,0.000126,0.002747,-0.000090,-0.000218,-0.007211,0.006033
Party Strength (angular),-0.003926,0.000239,0.004265,0.002811,-0.016580,-0.000993,0.000055,0.000127,-0.001279,0.000030,0.003030,0.000930,0.005435
Party Strength (mahalanobis_unweighted),-0.024297,-0.012776,-0.008674,-0.026532,0.033064,0.063440,-0.000228,-0.001344,-0.012632,-0.006131,-0.004493,0.040573,-0.038252
Party Strength (DM_L1_BONUS),0.006398,-0.001513,-0.013935,0.001612,-0.036842,0.031728,0.000029,0.000361,-0.029069,0.000059,0.002828,0.026749,0.012593
Party Strength (DM_HYBRID),-0.003631,-0.000849,-0.002676,0.001585,-0.021384,0.013338,0.000057,0.000113,-0.001468,-0.000097,0.002238,0.006281,0.003904


In [106]:
df_party_strength_diffs.loc[[f"Party Visibility ({m})" for m in EVAL_DISTANCE_METHODS], BIG_PARTIES].style.background_gradient('inferno')

_party,SP,Grüne,GLP,Mitte,FDP,SVP
Party Strength (L2_sv),-0.047643,-0.054866,0.026246,0.060597,0.000069,-0.017773
Party Strength (L1),-0.028252,-0.035596,0.017515,0.038648,-0.001653,-0.009340
Party Strength (AC),-0.007211,-0.012346,-0.000935,0.002747,-0.002342,0.006033
Party Strength (angular),0.000930,-0.000993,-0.016580,-0.001279,0.002811,0.005435
Party Strength (mahalanobis_unweighted),0.040573,0.063440,0.033064,-0.012632,-0.026532,-0.038252
Party Strength (DM_L1_BONUS),0.026749,0.031728,-0.036842,-0.029069,0.001612,0.012593
Party Strength (DM_HYBRID),0.006281,0.013338,-0.021384,-0.001468,0.001585,0.003904


## Matching Method - Party Visibility Table

In [ ]:
df_candidates = df_candidates.load_candidate_recommendation_counts('df_candidates_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [9]:
df_party_visibilities = df_candidates.get_party_visibilities()

df_party_visibilities_pref = df_party_visibilities.join(
    df_voters[df_voters['_party']!='Keine']['_party'].value_counts(normalize=True)
).rename(columns={'proportion': 'Preferred Party'})

df_party_visibilities_pref

,Party Visibility (L2_sv),Party Visibility (L1),Party Visibility (AC),Party Visibility (angular),Party Visibility (mahalanobis_unweighted),Party Visibility (DM_L1_BONUS),Party Visibility (DM_HYBRID),Preferred Party
_party,,,,,,,,
Andere,0.073464,0.068634,0.076723,0.067122,0.046752,0.074424,0.067417,0.015305
Centre,0.218844,0.196895,0.160995,0.158310,0.147020,0.130583,0.158184,0.110131
EDU,0.028473,0.027791,0.027217,0.026969,0.014318,0.025580,0.026244,0.005692
EVP,0.084871,0.074691,0.060690,0.064592,0.053968,0.048707,0.059966,0.019180
FDP,0.079726,0.078578,0.077889,0.081963,0.053699,0.080764,0.080737,0.135220
GLP,0.150703,0.141973,0.129672,0.119243,0.157521,0.098981,0.114439,0.182699
Green,0.122854,0.142125,0.165375,0.172944,0.232599,0.200888,0.182498,0.142800
Lega,0.000599,0.000649,0.000602,0.000681,0.000423,0.000654,0.000682,0.000719
MCG,0.001414,0.001645,0.001888,0.001889,0.000540,0.002124,0.001880,0.000170


In [10]:
# calculate deviations of party strengths from preferred parties
df_party_visibilities_pref_diffs = pd.DataFrame()

for c in [c for c in df_party_visibilities_pref.columns if c.startswith('Party Visibility')]:
    df_party_visibilities_pref_diffs[f'{c} - Relative'] = (df_party_visibilities_pref[c] / df_party_visibilities_pref['Preferred Party']) - 1
    df_party_visibilities_pref_diffs[f'{c} - Absolute'] = df_party_visibilities_pref[c] - df_party_visibilities_pref['Preferred Party']

df_party_visibilities_pref_diffs

,Party Visibility (L2_sv) - Relative,Party Visibility (L2_sv) - Absolute,Party Visibility (L1) - Relative,Party Visibility (L1) - Absolute,Party Visibility (AC) - Relative,Party Visibility (AC) - Absolute,Party Visibility (angular) - Relative,Party Visibility (angular) - Absolute,Party Visibility (mahalanobis_unweighted) - Relative,Party Visibility (mahalanobis_unweighted) - Absolute,Party Visibility (DM_L1_BONUS) - Relative,Party Visibility (DM_L1_BONUS) - Absolute,Party Visibility (DM_HYBRID) - Relative,Party Visibility (DM_HYBRID) - Absolute
_party,,,,,,,,,,,,,,
Andere,3.799815,0.058158,3.484250,0.053328,4.012777,0.061418,3.385511,0.051817,2.054577,0.031446,3.862544,0.059118,3.404770,0.052112
Centre,0.987115,0.108712,0.787815,0.086763,0.461839,0.050863,0.437465,0.048179,0.334951,0.036889,0.185701,0.020452,0.436323,0.048053
EDU,4.002155,0.022781,3.882228,0.022098,3.781508,0.021525,3.737917,0.021277,1.515305,0.008625,3.493893,0.019888,3.610517,0.020552
EVP,3.424953,0.065691,2.894193,0.055511,2.164204,0.041510,2.367665,0.045412,1.813717,0.034787,1.539417,0.029526,2.126433,0.040785
FDP,-0.410401,-0.055495,-0.418891,-0.056643,-0.423987,-0.057332,-0.393856,-0.053257,-0.602878,-0.081521,-0.402724,-0.054457,-0.402926,-0.054484
GLP,-0.175128,-0.031996,-0.222915,-0.040726,-0.290240,-0.053027,-0.347329,-0.063457,-0.137811,-0.025178,-0.458231,-0.083718,-0.373623,-0.068261
Green,-0.139674,-0.019945,-0.004727,-0.000675,0.158092,0.022575,0.211094,0.030144,0.628851,0.089800,0.406779,0.058088,0.277997,0.039698
Lega,-0.167245,-0.000120,-0.097849,-0.000070,-0.163668,-0.000118,-0.053850,-0.000039,-0.411563,-0.000296,-0.091053,-0.000065,-0.052062,-0.000037
MCG,7.309759,0.001244,8.666660,0.001475,10.098160,0.001718,10.104209,0.001719,2.176519,0.000370,11.481272,0.001954,10.048763,0.001710


In [11]:
# cell coloring values (positive and negative deviations from preferred parties)

df_main_parties_strengths_pref_diffs = df_party_visibilities_pref_diffs[[c for c in df_party_visibilities_pref_diffs.columns if 'Relative' in c]].T[[p for p in PARTIES_LEFT_TO_RIGHT if p in df_party_visibilities.index and p not in ['Andere', 'Parteilos']]]

(100 * df_main_parties_strengths_pref_diffs / df_main_parties_strengths_pref_diffs.abs().max()).round()

_party,SP,Green,PdA,GLP,EVP,Centre,FDP,MCG,Lega,SVP,EDU
Party Visibility (L2_sv) - Relative,-100.0,-22.0,-12.0,-38.0,100.0,100.0,-68.0,64.0,-41.0,-62.0,100.0
Party Visibility (L1) - Relative,-85.0,-1.0,20.0,-49.0,85.0,80.0,-69.0,75.0,-24.0,-47.0,97.0
Party Visibility (AC) - Relative,-68.0,25.0,46.0,-63.0,63.0,47.0,-70.0,88.0,-40.0,-27.0,94.0
Party Visibility (angular) - Relative,-64.0,34.0,100.0,-76.0,69.0,44.0,-65.0,88.0,-13.0,-28.0,93.0
Party Visibility (mahalanobis_unweighted) - Relative,-35.0,100.0,-35.0,-30.0,53.0,34.0,-100.0,19.0,-100.0,-100.0,38.0
Party Visibility (DM_L1_BONUS) - Relative,-45.0,65.0,96.0,-100.0,45.0,19.0,-67.0,100.0,-22.0,-15.0,87.0
Party Visibility (DM_HYBRID) - Relative,-61.0,44.0,82.0,-82.0,62.0,44.0,-67.0,88.0,-13.0,-30.0,90.0


In [94]:
# copy them to clipboard for chatgpt

(100 * df_main_parties_strengths_pref_diffs / df_main_parties_strengths_pref_diffs.abs().max()).round()

In [95]:
df = df_party_visibilities.T

df['Distance Metric'] = [x.split('(')[1].split(')')[0] for x in df.index]
df['Distance Metric'] = df['Distance Metric'].map(lambda x: DISTANCE_METHODS2NAMES.get(x, x))
df[[p for p in PARTIES_LEFT_TO_RIGHT if p in df_party_visibilities.index]] = 100 * np.round(df[[p for p in PARTIES_LEFT_TO_RIGHT if p in df_party_visibilities.index]], 4)

df_method_party_strength = df[['Distance Metric'] + [p for p in PARTIES_LEFT_TO_RIGHT if p in df_party_visibilities.index and p not in ['Andere', 'Parteilos']]]

df_method_party_strength

_party,Distance Metric,SP,Grüne,PdA,GLP,EVP,Mitte,FDP,MCG,Lega,SVP,EDU
Party Strength (L2_sv),L2,14.79,12.29,0.69,15.07,8.49,21.88,7.97,0.14,0.06,6.43,2.85
Party Strength (L1),L1,16.73,14.21,0.83,14.20,7.47,19.69,7.86,0.16,0.06,7.27,2.78
Party Strength (AC),Agreement Count,18.84,16.54,0.95,12.97,6.07,16.10,7.79,0.19,0.06,8.32,2.72
Party Strength (angular),Angular,19.38,17.29,1.20,11.92,6.46,15.83,8.20,0.19,0.07,8.26,2.70
Party Strength (mahalanobis_unweighted),Mahalanobis,23.17,23.26,0.58,15.75,5.40,14.70,5.37,0.05,0.04,4.38,1.43
Party Strength (DM_L1_BONUS),L1 Bonus,21.78,20.09,1.17,9.90,4.87,13.06,8.08,0.21,0.07,8.97,2.56
Party Strength (DM_HYBRID),Hybrid,19.74,18.25,1.12,11.44,6.00,15.82,8.07,0.19,0.07,8.16,2.62


#### Adjusted Vote Shares of Parties

In [8]:
df_candidates = df_candidates.load_candidate_recommendation_counts('df_candidates_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])
df_candidates = df_candidates.add_n_votes()

df_candidates.get_party_visibilities()['Party Visibility (Actual Votes)']

/Users/dustinbrunner/Programming/ETH/FS24/recommender-systems-for-politics/rsfp/data.py:1172: UserWarning: Votes have already been added to the DataFrame. Skipping.
  warnings.warn("Votes have already been added to the DataFrame. Skipping.")


_party
Andere       0.022769
Centre       0.146591
EDU          0.011307
EVP          0.019940
FDP          0.138143
GLP          0.080449
Green        0.104057
Lega         0.002239
MCG          0.003296
Parteilos    0.004658
PdA          0.004245
SP           0.191439
SVP          0.270867
Name: Party Visibility (Actual Votes), dtype: float64

## Bias Metrics (BIA)

In [10]:
df_candidates = df_candidates.load_candidate_recommendation_counts('df_candidates_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [21]:
df_party_visibilities = df_candidates.get_party_visibilities()[[f"Party Visibility ({m})" for m in EVAL_DISTANCE_METHODS]]
df_party_visibilities

,Party Visibility (L2_sv),Party Visibility (L1),Party Visibility (AC),Party Visibility (angular),Party Visibility (mahalanobis_unweighted),Party Visibility (DM_L1_BONUS),Party Visibility (DM_HYBRID)
_party,,,,,,,
Andere,0.073464,0.068634,0.076723,0.067122,0.046752,0.074424,0.067417
Centre,0.218844,0.196895,0.160995,0.158310,0.147020,0.130583,0.158184
EDU,0.028473,0.027791,0.027217,0.026969,0.014318,0.025580,0.026244
EVP,0.084871,0.074691,0.060690,0.064592,0.053968,0.048707,0.059966
FDP,0.079726,0.078578,0.077889,0.081963,0.053699,0.080764,0.080737
GLP,0.150703,0.141973,0.129672,0.119243,0.157521,0.098981,0.114439
Green,0.122854,0.142125,0.165375,0.172944,0.232599,0.200888,0.182498
Lega,0.000599,0.000649,0.000602,0.000681,0.000423,0.000654,0.000682
MCG,0.001414,0.001645,0.001888,0.001889,0.000540,0.002124,0.001880


In [18]:
# RELATIVE VISIBILITY CHANGES FOR EACH PARTY USING A DISTANCE METHOD IN COMPARISON TO THE MEDIAN VISIBILITY OF PARTY OVER ALL OTHER EVALUATED METHODS

df_party_visibilities_diffs = []
for c in df_party_visibilities.columns:
    df_party_visibilities_diffs.append(df_party_visibilities[c] / df_party_visibilities.drop(columns=c).median(axis=1) - 1)

df_party_visibilities_diffs = pd.concat(df_party_visibilities_diffs, axis=1)
df_party_visibilities_diffs.columns = df_party_visibilities.columns
df_party_visibilities_diffs

,Party Visibility (L2_sv),Party Visibility (L1),Party Visibility (AC),Party Visibility (angular),Party Visibility (mahalanobis_unweighted),Party Visibility (DM_L1_BONUS),Party Visibility (DM_HYBRID)
_party,,,,,,,
Andere,0.079942,-0.025649,0.127858,-0.055261,-0.341974,0.094056,-0.051112
Centre,0.382924,0.244222,0.017361,-0.008017,-0.079123,-0.182079,-0.009195
EDU,0.070159,0.044502,0.022954,0.008923,-0.471543,-0.055848,-0.031346
EVP,0.406838,0.238092,-0.025514,0.070691,-0.138465,-0.222453,-0.042714
FDP,0.000862,-0.020607,-0.029196,0.035517,-0.330696,0.020366,0.020021
GLP,0.210882,0.140733,-0.007160,-0.122072,0.265662,-0.271250,-0.157441
Green,-0.308723,-0.200292,-0.069466,-0.005706,0.375030,0.187563,0.078849
Lega,-0.080387,0.033818,-0.076437,0.088477,-0.350188,0.045679,0.090535
MCG,-0.249579,-0.127043,0.071488,0.072072,-0.713141,0.205022,0.064181


In [19]:
# BIA1 AVERAGE ABSOLUTE VALUE OF RELATIVE DIFFERENCES OVER 8 LARGEST PARTIES
df_party_visibilities_diffs.loc[LARGE_PARTIES].abs().mean()

Party Visibility (L2_sv)                     0.230071
Party Visibility (L1)                        0.143339
Party Visibility (AC)                        0.035842
Party Visibility (angular)                   0.040776
Party Visibility (mahalanobis_unweighted)    0.292372
Party Visibility (DM_L1_BONUS)               0.155349
Party Visibility (DM_HYBRID)                 0.052840
dtype: float64

In [20]:
# BIA2 MAXIMUM ABSOLUTE VALUE OF RELATIVE DIFFERENCES OVER 8 LARGEST PARTIES - SEARCH FOR PARTY IN RELATIVE CHANGES TABLE
df_party_visibilities_diffs.loc[LARGE_PARTIES].abs().max()

Party Visibility (L2_sv)                     0.406838
Party Visibility (L1)                        0.244222
Party Visibility (AC)                        0.078214
Party Visibility (angular)                   0.122072
Party Visibility (mahalanobis_unweighted)    0.471543
Party Visibility (DM_L1_BONUS)               0.271250
Party Visibility (DM_HYBRID)                 0.157441
dtype: float64